<p style="text-align:center">
    <img src="https://media.licdn.com/dms/image/v2/D560BAQGRKQWaOldXvg/company-logo_200_200/company-logo_200_200/0/1723551442229/cambo_ai_logo?e=2147483647&v=beta&t=FrK5k676UJMjoOULl4R3euChg1VvClew28fxNscSd9E" width="80" alt="CamboAI Logo">
</p>

# **I. Objectives**

**Estimated time needed: 60 minutes**

**After completing this lab you will be able to:**
- Know how to use sql in MySQL & Postgresql Databases.
- Perform **CRUD** Operation in SQL.
- Perform ETL From **Source** (MySQL) to **Destination** (PostgreSQL) Database.

# **II. Import Libraries**

In [15]:
# install pymysql, psycopg2 libraries
%pip install pymysql psycopg2 --quiet

Note: you may need to restart the kernel to use updated packages.


In [16]:
# import libraries
import pymysql
import psycopg2
import pandas as pd

# filter warning
import warnings
warnings.filterwarnings('ignore')

# **III. Extract the Data from Source**

In [17]:
# step 1: extract data from MySQL
def extract_from_mysql(mysql_config, query):
    # connect to MySQL database
    connection = pymysql.connect(
        host=mysql_config['host'],
        user=mysql_config['user'],
        password=mysql_config['password'],
        database=mysql_config['database']
    )

    # execute query and fetch data into a DataFrame
    df = pd.read_sql(query, connection)

    # close MySQL connection
    connection.close()

    return df

# **IV. Transform the Data**

In [18]:
# step 2: transform the data
def transform_data(df):
    # rename columns to lowercase
    df.columns = [col.lower() for col in df.columns]

    # other transformations can be done here
    return df

# **V. Load the Data Into Target**

In [19]:

import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values


def load_to_postgres(postgres_config, df, table_name="tbl_students"):

    # Validate the table name
    if table_name not in ("tbl_students", "public.tbl_students"):
        raise ValueError(f"Unsupported table name: {table_name}")

    if df.empty:
        print("No records to load.")
        return 0

    columns = [
        "id", "first_name", "last_name", "birth_date",
        "address", "gender", "email", "created_at", "updated_at"
    ]

    # Validate DataFrame columns
    missing_columns = set(columns) - set(df.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Normalize missing Pandas values to None for PostgreSQL
    clean_df = df[columns].astype(object).where(df[columns].notna(), None)

    # Reject duplicate IDs within this batch
    if clean_df["id"].duplicated().any():
        raise ValueError("Duplicate student IDs found in the input DataFrame.")

    records = list(clean_df.itertuples(index=False, name=None))
    connection = None

    try:
        connection = psycopg2.connect(
            host=postgres_config["host"],
            port=postgres_config.get("port", 5432),
            user=postgres_config["user"],
            password=postgres_config["password"],
            dbname=postgres_config["database"]
        )

        with connection.cursor() as cursor:

            # Create the target table if it does not exist
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS public.tbl_students (
                    id INTEGER PRIMARY KEY,
                    first_name VARCHAR(100),
                    last_name VARCHAR(100),
                    birth_date DATE,
                    address VARCHAR(255),
                    gender CHAR(1),
                    email VARCHAR(100),
                    created_at TIMESTAMP NOT NULL,
                    updated_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
                )
            """)

            # Upsert: insert new IDs or update existing IDs
            insert_query = sql.SQL("""
                INSERT INTO public.tbl_students ({fields})
                VALUES %s
                ON CONFLICT (id) DO UPDATE SET
                    first_name = EXCLUDED.first_name,
                    last_name = EXCLUDED.last_name,
                    birth_date = EXCLUDED.birth_date,
                    address = EXCLUDED.address,
                    gender = EXCLUDED.gender,
                    email = EXCLUDED.email,
                    created_at = EXCLUDED.created_at,
                    updated_at = EXCLUDED.updated_at
            """).format(
                fields=sql.SQL(", ").join(
                    sql.Identifier(col) for col in columns
                )
            )

            execute_values(
                cursor,
                insert_query.as_string(connection),
                records,
                page_size=1000
            )

        connection.commit()
        print(f"Successfully loaded {len(records)} records.")
        return len(records)

    except Exception:
        if connection is not None:
            connection.rollback()
        raise

    finally:
        if connection is not None:
            connection.close()

# **VI. ETL Operations**

In [23]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [24]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

# main function to run the ETL process
def run_etl():
    mysql_config = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD', ''),
    'database': os.getenv('MYSQL_DATABASE')
    }

    # New dynamic PostgreSQL configuration
    postgres_config = {
        'host': os.getenv('POSTGRES_HOST', 'localhost'),
        'user': os.getenv('POSTGRES_USER'),
        'password': os.getenv('POSTGRES_PASSWORD', ''),
        'database': os.getenv('POSTGRES_DATABASE')
    }

    query = 'SELECT * FROM tbl_students'

    # ETL process
    print("Step 1: Extracting data from MySQL...")
    df = extract_from_mysql(mysql_config, query)

    print("Step 2: Transforming data...")
    df_transformed = transform_data(df)

    print("Step 3: Loading data into PostgreSQL...")
    load_to_postgres(postgres_config, df_transformed, 'public.tbl_students')

    print("Step 4: ETL process completed.")

# Run the ETL process
if __name__ == '__main__':
    run_etl()

Step 1: Extracting data from MySQL...
Step 2: Transforming data...
Step 3: Loading data into PostgreSQL...
Successfully loaded 3 records.
Step 4: ETL process completed.


# Excellent! You have just completed this Notebook.

This notebook was created by [Chanthan DINA](https://www.linkedin.com/in/chanthandina/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!

## <h3 align="center"> &#169; CamboAI. All rights reserved. <h3/>